# Phase 2A — Forum → summary dengan state yang bertahan
Mode awal menggunakan fixture agar alur bisa diperiksa tanpa model. **Output fixture bukan bukti kemampuan Ollama.** Untuk pengujian AI asli: jalankan `ollama pull gemma4:e4b`, jalankan server Ollama, atur model/URL di `backend/.env`, lalu ubah `RUN_LIVE_OLLAMA=True`. Jika perangkat terlalu berat, gunakan `gemma4:e2b` atau `gemma3:4b`. Thinking dimatikan karena tugasnya hanya ekstraksi JSON.

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend' / 'jakroute').is_dir()), None)
assert ROOT, 'Buka notebook dari folder paket yang sudah diekstrak lengkap.'
sys.path.insert(0, str(ROOT / 'backend'))
DATA = ROOT / 'backend' / 'data'
from jakroute.providers import load_json
from jakroute.config import Settings

from jakroute.forum_state import ForumStore,OllamaSummarizer,DemoSummarizer
# False keeps offline verification deterministic. Set True after Ollama is ready.
RUN_LIVE_OLLAMA=False
settings=Settings.from_env()
processor=OllamaSummarizer(settings) if RUN_LIVE_OLLAMA else DemoSummarizer()
work=tempfile.TemporaryDirectory(prefix='jakroute_forum_')
store=ForumStore(Path(work.name)/'forum.sqlite3')
reports=load_json(DATA/'forum_reports.json')
print('OLLAMA LIVE' if RUN_LIVE_OLLAMA else 'FIXTURE SIMULATION — Ollama belum dipanggil')

FIXTURE SIMULATION — Ollama belum dipanggil


In [2]:
evaluation=[]
for report in reports[:3]:
 result=processor.summarize(report,store.snapshot())
 store.update(report,result)
 evaluation.append({'report_id':report['report_id'],'extraction':result,'version':store.snapshot()['version']})
 print(json.dumps(evaluation[-1],ensure_ascii=False,indent=2))
active=store.snapshot()['incidents']
assert any(i['resource_id']=='escalator_link' and i['effect']=='unavailable' for i in active)
assert active and all(i['status']!='resolved' for i in active)
assert evaluation[2]['extraction']['resolution_claimed'] is True
print('PASS: klaim penumpang tidak membuka kembali eskalator.')

{
  "report_id": "report_001",
  "extraction": {
    "summary": "Eskalator menuju peron rusak; akses eskalator tidak dapat digunakan.",
    "category": "failure",
    "effect": "unavailable",
    "severe": false,
    "resolution_claimed": false
  },
  "version": 1
}
{
  "report_id": "report_002",
  "extraction": {
    "summary": "Eskalator masih rusak.",
    "category": "failure",
    "effect": "unavailable",
    "severe": false,
    "resolution_claimed": false
  },
  "version": 2
}
{
  "report_id": "report_003",
  "extraction": {
    "summary": "Penumpang menduga eskalator telah diperbaiki; belum ada konfirmasi petugas.",
    "category": "failure",
    "effect": "none",
    "severe": false,
    "resolution_claimed": true
  },
  "version": 3
}
PASS: klaim penumpang tidak membuka kembali eskalator.


In [3]:
report=reports[3]
result=processor.summarize(report,store.snapshot())
store.update(report,result)
assert any(i['resource_id']=='north' and i['routing_code']==-1 for i in store.snapshot()['incidents'])
print(json.dumps(store.snapshot(),ensure_ascii=False,indent=2))

{
  "version": 4,
  "summary": "Eskalator masih rusak. Api dan asap pekat dilaporkan di koridor utara; jalur diblokir sementara.",
  "incidents": [
    {
      "incident_id": "escalator_link:failure",
      "resource_id": "escalator_link",
      "category": "failure",
      "summary": "Eskalator masih rusak.",
      "effect": "unavailable",
      "routing_code": 0,
      "status": "awaiting_officer_confirmation",
      "evidence_ids": [
        "report_001",
        "report_002",
        "report_003"
      ],
      "last_report_at": "2026-09-07T08:20:00Z",
      "resolution_claimed": true
    },
    {
      "incident_id": "north:hazard",
      "resource_id": "north",
      "category": "hazard",
      "summary": "Api dan asap pekat dilaporkan di koridor utara; jalur diblokir sementara.",
      "effect": "blocked",
      "routing_code": -1,
      "status": "active",
      "evidence_ids": [
        "report_004"
      ],
      "last_report_at": "2026-09-07T08:25:00Z",
      "resolution_cla

In [4]:
# Otoritas petugas disimulasikan di notebook; endpoint asli memeriksa token terpisah.
version=store.snapshot()['version']
key=next(i['incident_id'] for i in store.snapshot()['incidents'] if i['resource_id']=='escalator_link')
store.confirm(key,'petugas-simulasi','Perbaikan terverifikasi',version,'2026-09-07T08:30:00Z')
assert not any(i['resource_id']=='escalator_link' for i in store.snapshot()['incidents'])
assert any(i['resource_id']=='north' for i in store.snapshot()['incidents'])
# Restart reader: kondisi utara tetap tersimpan.
assert ForumStore(Path(work.name)/'forum.sqlite3').snapshot()==store.snapshot()
print('PASS: eskalator dibuka petugas, insiden utara tetap aktif.')

PASS: eskalator dibuka petugas, insiden utara tetap aktif.


In [5]:
if RUN_LIVE_OLLAMA:
 negative={'report_id':'negative_001','resource_id':'north','observed_at':'2026-09-07T08:40:00Z','message':'Tidak ada kebakaran; tadi hanya latihan evakuasi.'}
 result=processor.summarize(negative,store.snapshot())
 print('Uji negasi:',result)
 assert result['severe'] is False, 'Model salah membaca negasi; perbaiki prompt/model sebelum live.'
else:
 print('SKIPPED: uji kemampuan bahasa/negasi membutuhkan Ollama asli.')
work.cleanup()

SKIPPED: uji kemampuan bahasa/negasi membutuhkan Ollama asli.
